In [1]:
import xarray as xr, netCDF4 as nc, numpy as np, pandas as pd, os
from pathlib import Path

import dask
import dask.array as da
from dask.distributed import LocalCluster, Client
from datetime import datetime

In [2]:
os.chdir('/g/data/ng72/ms5578/ID_HW_BARRA')
workingDir = Path().absolute()
print(f"{workingDir}")

/g/data/ng72/ms5578/ID_HW_BARRA


In [3]:
fpath = "/g/data/ob53/BARRA2/output/reanalysis/AUS-11/BOM/ERA5/historical/hres/BARRA-R2/v1/fx/sftlf/latest/sftlf_AUS-11_ERA5_historical_hres_BOM_BARRA-R2_v1.nc"
write_path = f'{workingDir}/data/preprocess/'

In [4]:
ls_file = xr.open_dataset(fpath)
ls_ratio = ls_file.sftlf

ls_ratio

<xarray.DataArray 'sftlf' (lat: 646, lon: 1082)> Size: 3MB
[698972 values with dtype=float32]
Coordinates:
  * lat      (lat) float64 5kB -57.97 -57.86 -57.75 -57.64 ... 12.76 12.87 12.98
  * lon      (lon) float64 9kB 88.48 88.59 88.7 88.81 ... 207.2 207.3 207.4
    crs      int32 4B ...
Attributes:
    long_name:      Percentage of the grid  cell occupied by land (including ...
    standard_name:  land_area_fraction
    units:          %
    frequency:      fx
    grid_mapping:   crs

In [5]:
    ls_mask = ls_ratio.where(ls_ratio > 0.5, 0)
    ls_mask = ls_mask.where(ls_ratio <= 0.5, 1).astype(int)

In [9]:
ls_file = ls_mask.to_dataset()

In [11]:
encoding = {v: {"zlib": True, "complevel": 4, "shuffle": True} for v in ls_file.data_vars}

ls_file.to_netcdf(f'{write_path}land_sea_mask.nc',
                   encoding=encoding,
                   engine="netcdf4")